<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/947_EAASv3_Reporting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EaaS v3 Reporting Layer — Architecture Review

## What This Module Does

This module translates the evaluation engine’s results into an **executive-readable report**.

The orchestrator pipeline up to this point has produced:

* scenario-level scores
* run-level metrics
* trend analysis
* trigger alerts

Those outputs are extremely valuable technically — but they are not yet **decision-ready**.

The reporting module solves that problem by packaging the results into a structured **Markdown report** that highlights:

* overall system health
* operational reliability
* risk signals
* trend changes
* scenario-level diagnostics

In other words, it converts evaluation data into **actionable operational intelligence**.

---

# Where Reporting Fits in the Orchestrator

Within the EaaS agent pipeline:

```
Data Loading
      ↓
Scenario Scoring
      ↓
Trend / Trigger Analysis
      ↓
Executive Report
```

This module implements the **final stage**.

Everything upstream prepares the information.
This stage **presents it clearly**.

---

# Report Directory Management

## `_ensure_reports_dir`

This function ensures that a reports directory exists before saving the report.

```python
reports_root.mkdir(parents=True, exist_ok=True)
```

This prevents a common operational failure:

```
FileNotFoundError: directory does not exist
```

By handling this automatically, the system becomes more robust and easier to run in different environments.

Examples:

* local development
* CI pipelines
* scheduled evaluation jobs

This kind of defensive infrastructure code is small but extremely valuable.

---

# Trigger Flag Formatting

## `_format_trigger_flags`

This helper converts a list of trigger flags into readable text.

Example:

```
["policy_violation_trigger", "latency_trigger"]
```

becomes

```
policy_violation_trigger, latency_trigger
```

or

```
None
```

if no triggers exist.

This keeps the final report clean and readable.

---

# Core Report Generation

## `build_report_markdown`

This function assembles the entire evaluation report.

Rather than dumping raw JSON data, the function organizes the report into **clear operational sections**.

This structure mirrors how real reliability reports are written.

---

# Report Structure

The report follows a very logical layout.

## 1. Headline Summary

```
Evaluation score
System status
Overall pass rate
Outcome accuracy
```

This section gives leadership an **instant snapshot** of system health.

A leader can glance at this section and quickly understand whether the system is performing well.

Example interpretation:

| Metric           | Meaning               |
| ---------------- | --------------------- |
| Evaluation score | overall reliability   |
| System status    | operational health    |
| Pass rate        | scenario success rate |
| Outcome accuracy | customer impact       |

---

## 2. Functional Accuracy

This section breaks down the three dimensions of system correctness:

```
Issue classification accuracy
Resolution path accuracy
Outcome accuracy
```

This is extremely useful for engineering teams.

If performance declines, the team can quickly determine whether the issue lies in:

* problem detection
* routing logic
* outcome handling

---

## 3. Risk & Safety

```
High-risk failures
Weighted failure rate
Human review rate
```

This section highlights **business risk exposure**.

It reflects one of the strongest design aspects of your scoring system — risk-aware evaluation.

For example:

```
High-risk failures: 2
Weighted failure rate: 0.31
```

immediately signals that serious issues may be occurring.

---

## 4. Operational Performance

```
Average latency
P95 latency
```

These metrics evaluate **system responsiveness**.

Latency is a critical operational indicator because poor response time directly affects:

* customer satisfaction
* system scalability
* automation efficiency

Including both average and P95 latency aligns the system with industry-standard reliability monitoring practices.

---

## 5. Trend & Stability

This section communicates **changes relative to previous runs**.

Example signals:

```
Regression flags
Improvement flags
Trigger flags
```

This helps teams answer an essential operational question:

> Is the system getting better or worse?

This is one of the features that transforms EaaS from a simple benchmark into a **continuous monitoring system**.

---

# Scenario Diagnostics

## Failure Table

The final section identifies **specific scenarios that failed**.

Example output:

| Scenario | Risk | Severity | Issue | Path | Outcome |
| -------- | ---- | -------- | ----- | ---- | ------- |

This is extremely valuable because it allows engineers to:

* reproduce the failure
* diagnose root causes
* verify fixes

Without this section, teams would know **that something failed** but not **what failed**.

---

# Report Saving

## `build_and_save_report`

This function writes the report to disk.

Example output path:

```
agents/output/eaas_v3_reports/
```

Example file:

```
eaas_v3_report_RUN_2026_01_24.md
```

This naming scheme ensures reports are:

* uniquely identifiable
* easy to locate
* easy to archive

In a real production environment, these reports could easily be:

* uploaded to dashboards
* attached to CI pipelines
* stored in monitoring systems

---

# What Stands Out Most

Several things make this reporting design strong.

---

## Clear Operational Story

The report structure tells a coherent story:

1. Overall system health
2. Functional correctness
3. Risk exposure
4. Operational performance
5. Stability trends
6. Specific failures

This flow mirrors how real system health reviews are conducted.

---

## Balanced Audience Design

The report serves **two audiences simultaneously**.

### Leadership

They care about:

```
evaluation_score
system_status
risk triggers
```

### Engineers

They care about:

```
scenario failures
classification accuracy
routing accuracy
latency
```

The report successfully communicates to both groups.

---

## Deterministic Reporting

Importantly, the report is built entirely from **deterministic metrics**.

No LLM interpretation occurs here.

This ensures that the report:

* is reproducible
* cannot hallucinate
* can be audited

For evaluation systems, that level of determinism is essential.

---

# Suggested Improvements

The module is already very strong, but a few enhancements could make it even better.

---

## 1. Sort Failures by Severity

Right now failures appear in dataset order.

Sorting by severity would surface the most important failures first.

Example:

```python
failing = sorted(
    failing,
    key=lambda s: s.get("severity_weight", 1),
    reverse=True
)
```

This prioritizes high-risk scenarios.

---

## 2. Add Scenario Failure Reason

You could add a column indicating **why the scenario failed**.

Example:

```
classification_error
routing_error
outcome_error
```

This would improve debugging.

---

## 3. Include Scenario Count Summary

Before the failure table, you might add:

```
Total scenarios evaluated: 48
Total failures: 3
```

This helps readers quickly gauge failure scope.

---

# Why CEOs Would Appreciate This Report

A business leader reviewing this report sees something very reassuring:

* the system evaluates AI performance continuously
* operational risks are clearly surfaced
* performance trends are monitored
* failures are traceable to specific scenarios

In other words, the AI system is **being governed responsibly**.

That level of transparency is rare in most AI deployments.

---

# Final Assessment

This reporting module is:

* cleanly designed
* operationally meaningful
* aligned with real reliability reporting practices

It successfully converts the output of the evaluation engine into **clear, structured intelligence** that both engineers and leadership can use.

Together with your scoring and trend analysis modules, it completes a **very credible AI reliability monitoring system**.




In [ ]:
"""Reporting utilities for EaaS v3.

Builds an executive-friendly markdown report for each evaluation run.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List

from datetime import datetime, timezone

from config import EvalAsServiceConfig


def _now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _ensure_reports_dir(project_root: Path, config: EvalAsServiceConfig) -> Path:
    reports_root = project_root / config.reports_dir
    reports_root.mkdir(parents=True, exist_ok=True)
    return reports_root


def _format_trigger_flags(trigger_flags: List[str]) -> str:
    if not trigger_flags:
        return "None"
    return ", ".join(sorted(trigger_flags))


def build_report_markdown(
    *,
    run_id: str,
    run_metrics: Dict[str, Any],
    trigger_summary: Dict[str, Any],
    scenario_scores: List[Dict[str, Any]],
) -> str:
    ts = _now_utc_iso()

    headline_score = run_metrics.get("evaluation_score", 0.0)
    overall_pass_rate = run_metrics.get("overall_pass_rate", 0.0)
    outcome_accuracy = run_metrics.get("outcome_accuracy", 0.0)
    issue_accuracy = run_metrics.get("issue_classification_accuracy", 0.0)
    path_accuracy = run_metrics.get("resolution_path_accuracy", 0.0)
    high_risk_failures = run_metrics.get("high_risk_failures", 0)
    human_review_rate = run_metrics.get("human_review_rate", 0.0)
    avg_latency = run_metrics.get("avg_latency_ms", 0.0)
    p95_latency = run_metrics.get("p95_latency_ms", 0.0)
    weighted_failure_rate = run_metrics.get("weighted_failure_rate", 0.0)
    regression_flags = run_metrics.get("regression_flags", [])
    improvement_flags = run_metrics.get("improvement_flags", [])
    trigger_flags = run_metrics.get("trigger_flags", [])

    system_status = trigger_summary.get("system_status", "unknown")

    lines: List[str] = []
    lines.append(f"# EaaS v3 Evaluation Report — {run_id}")
    lines.append("")
    lines.append(f"Generated at: `{ts}`")
    lines.append("")
    lines.append("## Headline")
    lines.append("")
    lines.append(f"- **Evaluation score:** {headline_score:.3f}")
    lines.append(f"- **System status:** `{system_status}`")
    lines.append(f"- **Overall pass rate:** {overall_pass_rate:.3f}")
    lines.append(f"- **Outcome accuracy:** {outcome_accuracy:.3f}")
    lines.append("")
    lines.append("## Functional Accuracy")
    lines.append("")
    lines.append(f"- **Issue classification accuracy:** {issue_accuracy:.3f}")
    lines.append(f"- **Resolution path accuracy:** {path_accuracy:.3f}")
    lines.append(f"- **Outcome accuracy:** {outcome_accuracy:.3f}")
    lines.append("")
    lines.append("## Risk & Safety")
    lines.append("")
    lines.append(f"- **High-risk failures:** {high_risk_failures}")
    lines.append(f"- **Weighted failure rate:** {weighted_failure_rate:.3f}")
    lines.append(f"- **Human review rate:** {human_review_rate:.3f}")
    lines.append("")
    lines.append("## Operational Performance")
    lines.append("")
    lines.append(f"- **Average latency (ms):** {avg_latency:.2f}")
    lines.append(f"- **P95 latency (ms):** {p95_latency:.2f}")
    lines.append("")
    lines.append("## Trend & Stability")
    lines.append("")
    lines.append(f"- **Regression flags:** {', '.join(regression_flags) if regression_flags else 'None'}")
    lines.append(f"- **Improvement flags:** {', '.join(improvement_flags) if improvement_flags else 'None'}")
    lines.append(f"- **Trigger flags (current run):** {_format_trigger_flags(trigger_flags)}")
    lines.append("")
    lines.append("## Scenario Diagnostics (Top Failures)")
    lines.append("")

    failing = [
        s
        for s in scenario_scores
        if not (
            s.get("issue_classification_correct")
            and s.get("resolution_path_correct")
            and s.get("outcome_correct")
        )
    ]
    if not failing:
        lines.append("All scenarios passed for this run.")
    else:
        lines.append("| Scenario ID | Business Risk | Severity | Issue Correct | Path Correct | Outcome Correct |")
        lines.append("| ----------- | ------------- | -------- | ------------- | ------------ | --------------- |")
        for s in failing:
            lines.append(
                f"| {s.get('scenario_id')} "
                f"| {s.get('business_risk', 'n/a')} "
                f"| {s.get('severity_weight', 1.0)} "
                f"| {s.get('issue_classification_correct')} "
                f"| {s.get('resolution_path_correct')} "
                f"| {s.get('outcome_correct')} |"
            )

    return "\n".join(lines)


def build_and_save_report(
    *,
    project_root: str,
    config: EvalAsServiceConfig,
    run_id: str,
    run_metrics: Dict[str, Any],
    trigger_summary: Dict[str, Any],
    scenario_scores: List[Dict[str, Any]],
) -> str:
    root = Path(project_root)
    reports_root = _ensure_reports_dir(root, config)

    safe_run_id = run_id or "unknown_run"
    filename = f"eaas_v3_report_{safe_run_id}.md"
    report_path = reports_root / filename

    markdown = build_report_markdown(
        run_id=safe_run_id,
        run_metrics=run_metrics,
        trigger_summary=trigger_summary,
        scenario_scores=scenario_scores,
    )

    with report_path.open("w", encoding="utf-8") as f:
        f.write(markdown)

    return str(report_path)

